# 1. Preparación de los datos

## 1.1 Preparación e inspección

In [182]:
import pandas as pd
import numpy as np
import random as rd
import seaborn as sns
import openpyxl as opx
import plotly.express as px
from IPython.display import display, HTML

In [183]:
# ------------------------------------------
# 1.1.1 Descargar los datos, fijar semilla
# ------------------------------------------
rd.seed(1234)

archivo = "2026 BASE RESERVAS.xlsx"

df1 = pd.read_excel(archivo, sheet_name="EMISION")
df2 = pd.read_excel(archivo, sheet_name="SINIESTRALIDAD")

In [184]:
# ------------------------------------------
# 1.1.2 Revisar y limpiar variables
# ------------------------------------------

tablas = []

for nombre, df in [("Emisión", df1), ("Siniestros", df2)]:

    resumen = pd.DataFrame({
        "Tipo": df.dtypes,
        "Nulos": df.isna().sum()
    })

    tabla = f"""
    <div style="flex:1; min-width:0;">
        <h3>{nombre}</h3>
        <p>Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas</p>
        <p>Duplicados: {df.duplicated().sum()}</p>
        {resumen.to_html()}
    </div>
    """

    tablas.append(tabla)

display(HTML(
    '<div style="display:flex; gap:30px; align-items:flex-start;">'
    + "".join(tablas)
    + '</div>'
))

,Tipo,Nulos
año,int64,0
tipo_embarcacion,str,0
uso,str,0
anio_construccion,int64,0
prima_emitida,int64,0
prima_retenida,int64,0
comision_directa,int64,0
,Tipo,Nulos
año,int64,0
tipo_embarcacion,str,0


In [185]:
# --------------------------------------------
# 1.1.3 Corrección de tipos y limpieza de datos
# --------------------------------------------

# Emision 
emision = df1.copy()
emision = emision.dropna()
emision['tipo_embarcacion'] = emision['tipo_embarcacion'].astype('category')
emision['uso'] = emision['uso'].astype('category')
emision['anio_construccion'] = emision['anio_construccion'].astype('Int64')

# Siniestros
siniestros = df2.copy()
siniestros['tipo_embarcacion'] = siniestros['tipo_embarcacion'].astype('category')
siniestros['uso'] = siniestros['uso'].astype('category')
siniestros['anio_construccion'] = siniestros['anio_construccion'].astype('Int64')
siniestros.drop(columns=['cobertura'], inplace=True)
siniestros = siniestros.dropna()


In [186]:
# ------------------------------------------
# 1.1.4 Cantidad de valores negativos
# ------------------------------------------

tablas_negativos = []

for nombre, df in [("Emisión", emision), ("Siniestros", siniestros)]:

    resumen = pd.DataFrame({
        "variable": df.columns,
        "Negativos": 0,
        "Ceros": 0
    })

    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):

            resumen.loc[
                resumen["variable"] == column, "Negativos"
            ] = (df[column] < 0).sum()

            resumen.loc[
                resumen["variable"] == column, "Ceros"
            ] = (df[column] == 0).sum()

    tabla = f"""
    <div style="flex:1; min-width:0;">
        {resumen.to_html(index=False)}
    </div>
    """

    tablas_negativos.append(tabla)

display(HTML(
    '<div style="display:flex; gap:30px; align-items:flex-start;">'
    + "".join(tablas_negativos)
    + '</div>'
))

variable,Negativos,Ceros
año,0,0
tipo_embarcacion,0,0
uso,0,0
anio_construccion,0,0
prima_emitida,1126,0
prima_retenida,938,2009
comision_directa,857,2226
variable,Negativos,Ceros
año,0,0
tipo_embarcacion,0,0


In [187]:
# ------------------------------------------
# 1.1.5 Tipos de embarcación y uso
# ------------------------------------------

tipos_embarcacion = emision['tipo_embarcacion'].unique()
tipos_uso = emision['uso'].unique()

tipos_embarcacion_siniestros = siniestros['tipo_embarcacion'].unique()
tipos_uso_siniestros = siniestros['uso'].unique()

tabla_emision = pd.DataFrame({"Tipo de embarcación": tipos_embarcacion}).sort_values(by="Tipo de embarcación")
tabla_siniestros = pd.DataFrame({"Tipo de embarcación": tipos_embarcacion_siniestros}).sort_values(by="Tipo de embarcación")
tabla_emision_uso = pd.DataFrame({"Uso": tipos_uso}).sort_values(by="Uso")
tabla_siniestros_uso = pd.DataFrame({"Uso": tipos_uso_siniestros}).sort_values(by="Uso")


display(HTML(
    f"""
    <div style="display:flex; gap:30px; align-items:flex-start;">
        <div style="flex:1;">
            <h3>Emisión</h3>
            {tabla_emision.to_html(index=False)}
        </div>
        <div style="flex:1;">
            <h3>Siniestros</h3>
            {tabla_siniestros.to_html(index=False)}
        </div>
        <div style="flex:1;">
            <h3>Emisión - Uso</h3>
            {tabla_emision_uso.to_html(index=False)}
        </div>
        <div style="flex:1;">
            <h3>Siniestros - Uso</h3>
            {tabla_siniestros_uso.to_html(index=False)}
        </div>
    </div>
    """
))

Tipo de embarcación
Abastecedor
Abastecedor
Artefactos navales
Atunero
Barcaza o Chalan
Barco-Escuela
Barco-Escuela
Barco-Hidrografico
Buque Frigorifico
Buque Transportador de Autos


hay categorias que se repiten hay que limpiar eso

In [188]:
# ------------------------------------------
# 1.1.6 Limpieza de categorias 
# ------------------------------------------

columnas_categoricas = ["tipo_embarcacion", "uso"]

for df in [emision, siniestros]:
    for columna in columnas_categoricas:
        df[columna] = (
            df[columna]
            .str.strip()                       # Quitar espacios extremos
            .str.replace(r"\s+", " ", regex=True) # Unificar espacios internos
            .str.lower()                       # Convertir a minúsculas
            .str.replace(r"[^a-z0-9 ]", "", regex=True) # Quitar caracteres especiales
            .str.strip()                       # Quitar espacios extremos
        )

### Presencia de valores negativos

Los importes negativos podrían corresponder a cancelaciones de pólizas, devoluciones o ajustes contables. Lo correcto no seria eliminar estos registros negativos ya que pueden representar algo importante.

Al sumar las primas positivas y negativas obtenemos el monto neto, esto significa que nuestro analisis será en terminos netos monetarios ajustados

Existen tipos de embarcación con primas agregadas negativas en determinados años. 

No hay valores faltantes.

Un cero puede representar la ausencia de un pago.

## 1.2 Actualización de cifras


### 1.1 Ajustar los valores Mediante el INPC

Pasos:
1. Configurar el token de Banxico 
2. Descargar la serie mensual del INPC
3. Ajustar los valores mediante el INPC


In [ ]:
import banxicolab as sie

# 1.2.1 Configurar el token de Banxico
#sie.guardar_token()

In [190]:
# 1.2.2 Descargar la serie mensual del INPC
serie_inpc = sie.descargar("inpc", inicio="2015-01-01", fin="2025-12-31")["inpc"]
serie_inpc = pd.Series(serie_inpc)
serie_inpc


fecha
2015-01-01     87.110103
2015-02-01     87.275377
2015-03-01     87.630717
2015-04-01     87.403840
2015-05-01     86.967366
                 ...    
2025-08-01    140.867000
2025-09-01    141.197000
2025-10-01    141.708000
2025-11-01    142.645000
2025-12-01    143.042000
Name: inpc, Length: 132, dtype: float64

In [191]:
# 1.2.3. Ajustar las variables monetarias por inflación
from funciones_reservas import ajustar_inpc

#emision_ajustada
emision_ajustada = ajustar_inpc(
    df=emision,
    columnas=[
        "prima_emitida",
        "prima_retenida",
        "comision_directa"
    ],
    fecha_col="año",
    fecha_objetivo=2025,
    inpc=serie_inpc,
    frecuencia="anual",
    criterio_anual="diciembre"
)

#siniestros_ajustados
siniestros_ajustados = siniestros.copy()

# Por columna de monto pagado
for desarrollo in range(4): 

    # i_monto pagado
    columna = f"{desarrollo}_monto pagado"

    # Año correspondiente al pago
    siniestros_ajustados["año_pago"] = (
        siniestros_ajustados["año"] + desarrollo
    )

    # Solo ajustar pagos hasta 2025
    mascara = siniestros_ajustados["año_pago"] <= 2025

    datos_temp = ajustar_inpc(
        df=siniestros_ajustados.loc[mascara, ["año_pago", columna]],
        columnas=columna,
        fecha_col="año_pago",
        fecha_objetivo=2025,
        inpc=serie_inpc,
        frecuencia="anual",
        criterio_anual="diciembre"
    )

    siniestros_ajustados[f"{columna}_ajustada"] = (
        datos_temp[f"{columna}_ajustada"]
    )

siniestros_ajustados.drop(columns="año_pago", inplace=True)

display(emision_ajustada.head())
display(siniestros_ajustados.head())


,año,tipo_embarcacion,uso,anio_construccion,prima_emitida,prima_retenida,comision_directa,factor_inpc,prima_emitida_ajustada,prima_retenida_ajustada,comision_directa_ajustada
0,2015,lancha,comercial,2007,3219,3219,644,1.606368,5170.900076,5170.900076,1034.501289
1,2015,buquetanque,comercial,1992,332844,332844,66569,1.606368,534670.104079,534670.104079,106934.342090
2,2015,velero,privado yo placer,2012,25759,25759,5151,1.606368,41378.445190,41378.445190,8274.403943
3,2015,abastecedor,privado yo placer,2001,-37,-37,-7,1.606368,-59.435633,-59.435633,-11.244579
4,2015,lancha,pesca,2013,-194,-194,-31,1.606368,-311.635481,-311.635481,-49.797422


,año,tipo_embarcacion,uso,anio_construccion,causa_siniestro,0_monto pagado,1_monto pagado,2_monto pagado,3_monto pagado,0_monto pagado_ajustada,1_monto pagado_ajustada,2_monto pagado_ajustada,3_monto pagado_ajustada
0,2015,abastecedor,comercial,1979,Incendio,29665062.0,0.0,0.0,0.0,4.765302e+07,0.0,0.0,0.0
1,2015,abastecedor,comercial,2011,Averia particular,15502388.0,0.0,0.0,0.0,2.490255e+07,0.0,0.0,0.0
2,2015,buquetanque,comercial,2013,Varadura,10377414.0,0.0,0.0,0.0,1.666995e+07,0.0,0.0,0.0
3,2015,buquetanque,comercial,2014,Negligencia de la tripulacion,9721320.0,0.0,0.0,0.0,1.561602e+07,0.0,0.0,0.0
4,2015,remolcador,remolcador,2013,Averia particular,9668053.0,0.0,0.0,0.0,1.553046e+07,0.0,0.0,0.0


# 2. Análisis de Mercado

## 2.1. Análisis de la prima emitida


### 2.1.1 Comparación de la evolución del mercado

- ¿Cómo ha evulucionado el mercado de seguros de embarcaciones?
- ¿cómo se distribuyen las primas? 
- ¿qué proporción de ellas se retiene o destina a comisiones?

In [192]:
# --------------------------------------------------
# 2.1.1 ANÁLISIS DE LA PRIMA EMITIDA
# --------------------------------------------------

# 1. Agrupar los montos ajustados por año
columnas_monetarias = [
    "prima_emitida_ajustada",
    "prima_retenida_ajustada",
    "comision_directa_ajustada"
]

emision_anual = (
    emision_ajustada
    .groupby("año")[columnas_monetarias]
    .sum()
    .sort_index()
)

# 2. Calcular los indicadores anuales

# Tasas de crecimiento de las primas (%)
columnas_primas = [
    "prima_emitida_ajustada",
    "prima_retenida_ajustada"
]

emision_anual[["crecimiento_emitida", "crecimiento_retenida"]] = (
    emision_anual[columnas_primas].pct_change() * 100
)

# Proporción de comisiones respecto a la prima emitida (%)
emision_anual["%Com/Prima"] = (
    emision_anual["comision_directa_ajustada"]
    / emision_anual["prima_emitida_ajustada"] * 100
)

# 3. Normalizar las variables (año base 2015 = 100)
columnas_comparacion = columnas_primas + ["%Com/Prima"]

comparacion = emision_anual[columnas_comparacion].copy()

# Todas las series comienzan en 100 para comparar su evolución
comparacion = (
    comparacion
    .div(comparacion.loc[2015])
    .mul(100)
    .reset_index()
)

# 4. Visualizar la evolución relativa
fig = px.line(
    comparacion,
    x="año",
    y=columnas_comparacion,
    markers=True,
    title="Evolución relativa de primas y comisiones ",
    labels={
        "año": "Año",
        "value": "Índice ",
        "variable": "Indicador"
    }
)

fig.update_layout(
    hovermode="x unified",
    legend_title_text="Indicador"
)

fig.show()

Con fines comparativos, se normalizaron las variables tomando 2015 como año base (índice de 100), para observar sus cambios relativos y facilitar la comparación de los comportamientos a lo largo de los años.

Se observa una tendencia creciente en las primas emitidas, alcanzando un aumento aproximado del 150% en 2025 respecto a 2015 lo cual el aumento de este último año puede deberse a los conflictos entre Estados Unidos e Irán. También destaca una caída importante en 2020, posiblemente relacionada con la disminución de la actividad marítima durante la pandemia, lo que pudo afectar la contratación de seguros. 

Por otro lado, las primas retenidas se mantienen relativamente estables a pesar del crecimiento de las emitidas, esto podría deberse a una mayor participación del reaseguro, lo que implicaría una menor proporción de riesgos asumidos directamente por las aseguradoras. 

Finalmente, la proporción de comisiones aumentó durante 2020 y 2021, posiblemente porque las primas emitidas disminuyeron más que las comisiones, mientras que en los últimos años esta proporción se redujo, lo que sugiere un menor peso relativo de las comisiones dentro del mercado.

### 2.1.2 Distribución de las primas de embarcación



- ¿Cuáles aportan más?
- ¿Cuáles aportan menos?
- ¿Como cambian las participaciones con el tiempo?

In [193]:
# --------------------------------------------------
#  2.1.2 Distribución de prima por tipo de embarcación
# --------------------------------------------------

import plotly.express as px

# 1. Agrupar las primas por año y tipo de embarcación
participacion = (
    emision_ajustada
    .groupby(["año", "tipo_embarcacion"])["prima_emitida_ajustada"]
    .sum()
    .unstack(fill_value=0)
)


# 2. Seleccionar los 6 tipos con mayor prima acumulada
principales = (
    participacion.sum()
    .nlargest(6)
    .index
)

# 3. Agrupar los demás tipos en una sola categoría
participacion["Otros"] = (
    participacion.drop(columns=principales).sum(axis=1)
)

participacion = participacion[
    list(principales) + ["Otros"]
]

# 4. Calcular la participación porcentual anual
participacion = (
    participacion
    .div(participacion.sum(axis=1), axis=0)
    .mul(100)
)

# 5. Convertir a formato largo
participacion = (
    participacion
    .reset_index()
    .melt(
        id_vars="año",
        var_name="Tipo de embarcación",
        value_name="Participación (%)"
    )
)

# 6. Graficar
fig = px.area(
    participacion,
    x="año",
    y="Participación (%)",
    color="Tipo de embarcación",
    title="Distribución anual de primas por tipo de embarcación"
)

fig.update_layout(
    yaxis=dict(range=[0, 100], ticksuffix="%"),
    xaxis_title="Año",
    hovermode="x unified"
)

fig.show()

Con el objetivo de analizar cómo ha cambiado la composición del mercado, se calculó la participación porcentual de cada tipo de embarcación respecto a la prima emitida total de cada año.

Se observa que entre 2015 y 2022 los abastecedores tuvieron una participación importante, destacando especialmente en 2019, sin embargo, a partir de 2023 los buques tanque petroleros comenzaron a dominar el mercado, alcanzando aproximadamente el 77% de las primas emitidas en 2025. Este cambio podría estar relacionado con lo antes ya mensionado de las tensiónes entre Estados Unidos e Irán, con ello aumento de los riesgos en el transporte marítimo de petróleo, que han provocado incrementos en los costos de aseguramiento de ciertas rutas, lo que podría explicar una mayor concentración de primas en este tipo de embarcaciones.

Esto sugiere que el mercado se ha vuelto más dependiente del segmento petrolero, lo que podría representar una mayor exposición a los riesgos asociados con esta actividad y una menor diversificación de las primas emitidas.

## 2.2 Análisis del monto de siniestros

### 2.2.1 Distribución del monto de siniestros por causa y cuanto cambia por año

- ¿Cuándo ocurren las mayores perdidas?
- ¿Cuáles son las principales causas de perdida?


In [194]:
# --------------------------------------------------
# 2.2.1.  Distribución anual de pérdidas por causa
# --------------------------------------------------

import plotly.express as px

# 1. Identificar las columnas monetarias ajustadas
columnas_pagos = [
    f"{i}_monto pagado_ajustada" for i in range(4)
]

# 2. Calcular el monto total de cada registro
siniestros_analisis = siniestros_ajustados.copy()

siniestros_analisis["monto_total"] = (
    siniestros_analisis[columnas_pagos]
    .sum(axis=1, min_count=1)
)

# 3. Agrupar los siniestros por año y causa
siniestros_anual = (
    siniestros_analisis
    .groupby(["año", "causa_siniestro"], as_index=False)["monto_total"]
    .sum()
    .sort_values("año")
)

# 4. Construir la gráfica de barras apiladas
fig = px.bar(
    siniestros_anual,
    x="año",
    y="monto_total",
    color="causa_siniestro",
    barmode="relative",
    title="Distribución anual de siniestros por causa",
    labels={
        "año": "Año de origen",
        "monto_total": "Monto de siniestros ajustado (MXN)",
        "causa_siniestro": "Causa del siniestro"
    }
)

# 5. Configuración visual
fig.update_layout(
    xaxis=dict(dtick=1),
    yaxis=dict(tickformat=",.0f"),
    hovermode="x unified",
    legend_title_text="Causa del siniestro",
    height=600
)

fig.show()

Se observa que 2015 presenta el mayor monto de siniestros del periodo, con aproximadamente 4,200 millones de pesos, de los cuales al rededor de 3,800 millones son principalmente en fenómenos naturales, esto podría relacionarse con la intensa temporada de huracanes registrada ese año, especialmente con el paso del huracán Patricia, lo que refleja la exposición del mercado marítimo a eventos naturales capaces de generar pérdidas importantes. Posteriormente, los montos disminuyen, aunque vuelven a presentar incrementos en 2023 y 2024, con una mayor participación de distintas causas, como varaduras y encallamientos, lo que muestra que las pérdidas del mercado también dependen de riesgos operativos y no únicamente de fenómenos naturales.

## 2.3 Conclusión del Análisis de Mercado 

Se observa que el mercado de seguros de embarcaciones presenta un crecimiento general en las primas emitidas, aunque con fluctuaciones importantes, como la caída de 2020, posiblemente relacionada con la disminución de la actividad marítima durante la pandemia, lo que pudo afectar la contratación de seguros. Por otro lado, las primas se han concentrado cada vez más en los buques tanque petroleros, posiblemente debido a una mayor actividad o al incremento de los costos de aseguramiento en este sector por conflictos geopolíticos, lo que implica una mayor dependencia del mercado hacia este tipo de embarcaciones.

En cuanto a los siniestros, destaca 2015 por las elevadas pérdidas asociadas a fenómenos naturales, posiblemente relacionadas con la temporada de huracanes, especialemente el paso del huracán Patricia, mientras que en años posteriores se observa una mayor participación de causas operativas, como varaduras y encallamientos. 

En conjunto, los resultados muestran un mercado en crecimiento, pero con una importante concentración de primas y exposición a pérdidas extraordinarias, lo que resalta la importancia de una adecuada diversificación y gestión de riesgos.

# 3. Siniestralidad y Conglomerados